# 17.1 持续学习 (Continual Learning)

持续学习使模型在不遗忘旧知识的情况下学习新知识。

本节涵盖：灾难性遗忘、正则化方法、回放方法

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

torch.manual_seed(42)

class SimpleModel(nn.Module):
    def __init__(self, d=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, n_classes))
    def forward(self, x):
        return self.net(x)

model = SimpleModel()
ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad = False

x_task1 = torch.randn(16, 64)
y_task1 = torch.randint(0, 5, (16,))
x_task2 = torch.randn(16, 64)
y_task2 = torch.randint(5, 10, (16,))

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(10):
    loss_task2 = F.cross_entropy(model(x_task2), y_task2)
    ewc_loss = 0
    for p, p_ref in zip(model.parameters(), ref_model.parameters()):
        ewc_loss += ((p - p_ref) ** 2).sum()
    total_loss = loss_task2 + 0.5 * ewc_loss
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

task1_acc_after = (model(x_task1).argmax(dim=-1) == y_task1).float().mean()
task2_acc = (model(x_task2).argmax(dim=-1) == y_task2).float().mean()

print('=== Continual Learning ===')
print(f'\nProblem: Catastrophic forgetting')
print(f'  Learning Task 2 degrades Task 1 performance')
print(f'\nEWC (Elastic Weight Consolidation) defense:')
print(f'  Task 1 accuracy after learning Task 2: {task1_acc_after:.2%}')
print(f'  Task 2 accuracy: {task2_acc:.2%}')
print(f'\nKey methods:')
print(f'  EWC: penalize changes to important weights')
print(f'  Replay: mix old task data with new task data')
print(f'  Progressive nets: add new columns for new tasks')
print(f'\nKey: Continual learning balances stability (old) vs plasticity (new).')